# Bookey 3/3 — Whisper-turbo LoRA for Tanglish questions

Fine-tunes `openai/whisper-small` (the model Bookey already runs with faster-whisper) with
LoRA on the native Tanglish dataset. Targets stay in Tamil script (English words in Tamil script, as
native transcribers write them); Bookey's romanizer turns the output into the Latin Tanglish the
learner sees. The merged model is converted to CTranslate2 int8 for faster-whisper.

* Kaggle: GPU T4 (x1 is enough), Internet on, no secrets needed: data comes from notebook 1's output. ~3–5 h.

In [ ]:
!pip -q install "transformers>=4.44,<4.53" "peft>=0.12" accelerate "datasets>=2.20,<4" jiwer evaluate soundfile librosa ctranslate2 && pip -q uninstall -y torchao

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # two GPUs through DataParallel were slower here
BASE = "openai/whisper-small"
MAX_STEPS, BATCH, ACCUM, LR = 3000, 16, 1, 1e-4   # batch 8 runs out of memory on a 16 GB T4

In [ ]:
import torch
from datasets import Audio, load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE, language="tamil", task="transcribe")
import glob, os
from datasets import DatasetDict
parts = sorted(glob.glob("/kaggle/input/**/part-*.parquet", recursive=True))
assert parts, "attach the output of bookey-1-tanglish-dataset"
raw = load_dataset("parquet", data_files=parts, split="train")
ds = DatasetDict({n: raw.filter(lambda s, n=n: s == n, input_columns="split") for n in ("train", "test")})
ds = ds.cast_column("audio", Audio(sampling_rate=16_000))

def prepare(batch):
    a = batch["audio"]
    batch["input_features"] = processor.feature_extractor(a["array"], sampling_rate=16_000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

ds = ds.map(prepare, remove_columns=ds["train"].column_names, num_proc=2)
# Tamil script is token-heavy in Whisper's BPE (~130 tokens for a 6 s clip); the decoder max is 448.
ds = ds.filter(lambda labels: len(labels) <= 440, input_columns="labels")
print(ds)

In [ ]:
from dataclasses import dataclass
from peft import LoraConfig, get_peft_model

# fp32 weights: with fp16 weights under fp16 autocast the loss went to NaN in the first steps.
model = WhisperForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float32)
model.config.forced_decoder_ids = None
model.generation_config.language, model.generation_config.task = "tamil", "transcribe"
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05,
                                         target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]))
model.print_trainable_parameters()

@dataclass
class Collator:
    def __call__(self, features):
        inputs = processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labels = processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:, 0] == processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")).all():
            lab = lab[:, 1:]
        inputs["labels"] = lab
        return inputs

In [ ]:
import evaluate
import math
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback

class StopOnBrokenLoss(TrainerCallback):
    """Fail in minutes instead of burning GPU hours on a run that has stopped learning."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss")
        if loss is not None and (math.isnan(loss) or loss == 0 or loss > 50):
            raise RuntimeError(f"training broke at step {state.global_step}: loss={loss}")

cer = evaluate.load("cer")

def metrics(pred):
    ids = pred.label_ids; ids[ids == -100] = processor.tokenizer.pad_token_id
    hyp = processor.batch_decode(pred.predictions, skip_special_tokens=True)
    ref = processor.batch_decode(ids, skip_special_tokens=True)
    return {"cer": cer.compute(predictions=hyp, references=ref)}

trainer = Seq2SeqTrainer(
    model=model, data_collator=Collator(), train_dataset=ds["train"], eval_dataset=ds["test"].select(range(min(100, len(ds["test"])))),
    compute_metrics=metrics, callbacks=[StopOnBrokenLoss()],
    args=Seq2SeqTrainingArguments(
        output_dir="/kaggle/working/whisper_out", per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        learning_rate=LR, warmup_steps=200, max_steps=MAX_STEPS, fp16=True, eval_strategy="steps", eval_steps=500,
        save_steps=500, save_total_limit=2, logging_steps=25, predict_with_generate=True, generation_max_length=444,
        per_device_eval_batch_size=4, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}, remove_unused_columns=False, label_names=["labels"], report_to="none"),
)
print("baseline:", trainer.evaluate())
trainer.train(resume_from_checkpoint=any(p.startswith("checkpoint") for p in os.listdir("/kaggle/working/whisper_out")) if os.path.isdir("/kaggle/working/whisper_out") else False)
print("fine-tuned:", trainer.evaluate())

In [ ]:
# Merge LoRA → full model → CTranslate2 int8 (what Bookey's faster-whisper loads), kept in this notebook's output.
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/whisper_merged"); processor.save_pretrained("/kaggle/working/whisper_merged")
!ct2-transformers-converter --model /kaggle/working/whisper_merged --output_dir /kaggle/working/whisper_ct2 --quantization int8_float16 --copy_files tokenizer.json preprocessor_config.json --force
print("Done: download whisper_ct2 from this notebook's output and set FASTER_WHISPER_MODEL to it")